# notebook 07 — 道A は失敗：コンパクト性はベル与件に強制される（no-go）

notebook 06 の申し送りで、私は「**道A（設定空間を円から実数直線へ）が最も原理に忠実な
非コンパクト化かもしれない**」と書いた。この notebook はそれを検証し——**外れだった**ことを
正直に記録する。

検証の結果はむしろ強い否定的（no-go 的）結論である:

> **ベル相関の与件 `E(θ)=cos2θ/(2N)`（θ＝2設定の角度差）を保つ限り、二点相関は角度差の
> 周期・有界関数であり、距離の定義・`st`・設定の配置のいずれをいじっても非コンパクト空間は
> 出ない。** 非コンパクト化は与件の改変（道I）を要し、符号情報（道II）は非コンパクト化では
> なくローレンツ化（符号数の変更）である。

これは notebook 06（コンパクト性は cos2θ に内在）を、設定空間の自由度まで含めて閉じる結果。


## 0. セットアップ

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import eigh
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def mds(D):
    n=D.shape[0]; J=np.eye(n)-np.ones((n,n))/n
    B=-0.5*J@(D**2)@J; ev,V=eigh(0.5*(B+B.T))
    i=np.argsort(ev)[::-1]; return ev[i], V[:,i]

def signature(ev):
    pos=ev[ev>1e-6]
    if len(pos)==0: return (0,0)
    deg=int(np.sum(np.abs(pos-pos[0])<1e-3*pos[0]))
    return (deg, len(pos))

def neglog_dist(C):
    a=np.abs(C); 
    with np.errstate(divide="ignore"): D=-np.log(a)
    np.fill_diagonal(D,0.0)
    if (~np.isfinite(D)).any(): D[~np.isfinite(D)]=D[np.isfinite(D)].max()
    return D


## 1. 道A：設定を直線上に置く——配置を変えても周期性は壊れない

ベル相関は `E(θ)=cos2θ/(2N)`、`θ` は2設定の**角度差**。設定を直線上のどこに置こうと、
相関は差 `θ_i−θ_j` の関数であり `cos` ゆえ周期的。配置（一様・クラスタ・2クラスタ）を
変えて、すべてコンパクト（円に巻き戻る）ことを確認する。


In [2]:
rng=np.random.default_rng(0)
placements={
 "uniform line [0,50]":  np.linspace(0,50,200),
 "exponential cluster":  np.sort(rng.exponential(5,200)),
 "two separated clusters":np.concatenate([np.linspace(0,2,100), np.linspace(20,22,100)]),
}
print(f"{'placement':26s} {'(deg, #pos)':>14s}   reading")
for name,x in placements.items():
    C=np.cos(2*(x[:,None]-x[None,:]))/6
    ev,_=mds(neglog_dist(C))
    print(f"{name:26s} {str(signature(ev)):>14s}   compact (winds to circle)")
print("\n=> Placement is irrelevant. cos2(difference) is periodic; the image stays compact.")


placement                     (deg, #pos)   reading
uniform line [0,50]              (2, 175)   compact (winds to circle)
exponential cluster              (1, 139)   compact (winds to circle)
two separated clusters           (1, 118)   compact (winds to circle)

=> Placement is irrelevant. cos2(difference) is periodic; the image stays compact.


## 2. なぜ配置では破れないのか（与件そのものの帰結）

ベル与件は **二点相関の関数形** を `cos2(角度差)` に固定する。これは「設定 i,j のどんな対でも
`C_{ij}=cos2(θ_i−θ_j)/(2N)`」という主張であり、`θ` の住む空間（円か直線か）に関わらず、
`C` は差の周期・有界関数。notebook 06 で見たとおり、そのような `C` は `(cos2θ,sin2θ)` の
内積＝**ランク2のコンパクト・カーネル**。設定の台を直線に広げても、`cos2` が差を円に
巻き戻すので、像は円のまま。

**配置（道A）は相関の関数形を変えない以上、コンパクト性を破れない。**


In [3]:
# Direct confirmation: rank of C is 2 regardless of how settings are placed on the line.
for name,x in placements.items():
    C=np.cos(2*(x[:,None]-x[None,:]))/6
    print(f"  {name:26s}: rank(C)={np.linalg.matrix_rank(C, tol=1e-9)} (=2 => circle)")


  uniform line [0,50]       : rank(C)=2 (=2 => circle)
  exponential cluster       : rank(C)=2 (=2 => circle)
  two separated clusters    : rank(C)=2 (=2 => circle)


## 3. 抜け道の網羅的封鎖

「与件を保ったまま非コンパクトを出す」抜け道の候補を挙げ、すべて潰す。


### L1. 多数のベルペアの積 → `T^p`（既出）
notebook 04 で確認済み。積はトーラス＝コンパクト。**不可。**


### L2. 走る基準（running angle）：角度を累積して非有界にする
設定角を増分の累積 `Σ` で非有界にしても、相関が `cos2(累積差)` なら差は周期に戻る。


In [4]:
n=200; incr=np.full(n,0.3); ang=np.cumsum(incr)  # unbounded running angle
C=np.cos(2*(ang[:,None]-ang[None,:]))/6
ev,_=mds(neglog_dist(C))
print("L2 running angle, cos2(diff):", signature(ev), "-> still periodic. NO.")


L2 running angle, cos2(diff): (1, 111) -> still periodic. NO.


### L3. ホロノミー：経路に沿った `cos2` の積から距離を作る
一見、積が幾何減衰して非コンパクトな直線を生むように見える。だが**与件に矛盾する**ことを
示す。


In [5]:
n=200; x=np.arange(n); step=0.2
base=abs(np.cos(2*step))
absC_chain=base**np.abs(x[:,None]-x[None,:])      # product along a chain = exp decay
D=-np.log(absC_chain); np.fill_diagonal(D,0.0)
ev,V=mds(D)
print("L3 chain-product 'holonomy':", signature(ev),
      f" coord0~index corr={np.corrcoef(V[:,0],x)[0,1]:.3f}  (looks like non-compact R)")

# But does it obey the Bell datum? The datum says C_ij = cos2((j-i)*step)/2N for ANY pair.
ij=5
datum = abs(np.cos(2*ij*step))
chain = base**ij
print(f"\n   Bell datum for |i-j|={ij}: |cos2({ij}*step)| = {datum:.4f}")
print(f"   chain product base^{ij}          = {chain:.4f}")
print(f"   They DIFFER => L3 violates the two-point Bell datum for non-adjacent pairs.")
print("   L3 is exponential-correlation (notebook 05 B2 = ROUTE I) in disguise. NO.")


L3 chain-product 'holonomy': (1, 1)  coord0~index corr=-1.000  (looks like non-compact R)

   Bell datum for |i-j|=5: |cos2(5*step)| = 0.4161
   chain product base^5          = 0.6629
   They DIFFER => L3 violates the two-point Bell datum for non-adjacent pairs.
   L3 is exponential-correlation (notebook 05 B2 = ROUTE I) in disguise. NO.


### L4. 非定常相関：`C_{ij}` を差でなく `θ_i,θ_j` 別々に依存させる
与件は差の関数だと言っているので、別々依存は与件の改変。形式的に試すと積型になり、
やはりコンパクト（ランク有限のフーリエカーネル）。


In [6]:
th=np.linspace(0,2*np.pi,120,endpoint=False)
# e.g. C_ij = cos2(theta_i)cos2(theta_j)/36 (non-stationary, but a product)
C=np.outer(np.cos(2*th),np.cos(2*th))/36
ev,_=mds(neglog_dist(C))
print("L4 non-stationary product:", signature(ev),
      f" rank(C)={np.linalg.matrix_rank(C,tol=1e-9)} -> finite-rank, compact. NO.")


L4 non-stationary product: (1, 118)  rank(C)=1 -> finite-rank, compact. NO.


## 4. まとめ：道A は失敗、そして no-go 的結論

### 確認できたこと

| 主張 | 結果 | ラベル |
|---|---|---|
| 道A（設定を直線に配置）で非コンパクトは出るか | **否**。配置不問でコンパクト | **established** |
| 配置を変えても `rank(C)=2`（円）のまま | §1–2 で確認 | **established** |
| 抜け道 L1（積）/L2（走る角）/L3（ホロノミー）/L4（非定常） | すべて不可、または与件違反 | **established** |
| L3 は指数相関（道I）の偽装で、二点与件に矛盾 | §3 で数値的に矛盾を提示 | **established** |

### no-go 的な結論（誠実版）
**ベル与件 `E(θ)=cos2θ/(2N)`（θ＝角度差）を保つ限り、創発する空間はコンパクトである。**
理由：与件は二点相関を差の周期・有界関数に固定し、それはランク2のコンパクト・カーネル
（notebook 06）。距離・`st`・設定配置（道A）はこの円上の操作で、像をコンパクトより広く
できない。非コンパクト化は与件の改変（道I）を要し、符号情報（道II）は非コンパクト化では
なく符号数の変更（ローレンツ化）である。

### 前回の見通しの訂正（過大主張の防止）
- notebook 06 で「道A が最も原理に忠実な非コンパクト化かもしれない」と書いたが、**これは
  外れだった**。配置は関数形を変えないので周期性を破れない。訂正して記録する。

### 規律の自己点検（引き継ぎ書 §6）
- 自分の前回の見通し（道A 有望）を検証し、否定的結果として正直に確定させた。✅
- 有望に見えた L3 を、与件との矛盾を数値で示して棄却した（都合よく採用しない）。✅
- no-go を「証明」と過大に呼ばず、数値的に強く支持される構造的結論として提示。✅

### この結果が新土台に対して持つ意味（重要）
これは行き止まりではなく、**地図の確定**である（引き継ぎ書 §2-1）。非コンパクトな (3+1)
時空に至るには、`cos2θ` 与件のほかに **少なくとも一つの追加原理が論理的に必要**だと
判明した。その追加原理は次のいずれかの形を取る:

1. **与件の拡張（道I）**：`cos2θ` を、ある条件下で減衰／非周期な相関へ拡張する原理。
   §2-2「最小から最大へ登る」の枠で、この拡張が `st`／作用 `S=−βΣe^{−d}` の帰結として
   導けるかが次の問い。
2. **符号原理（道II）**：相関の符号を計量の符号に対応させる原理。これは非コンパクト化とは
   独立で、ローレンツ化を担う。`T^p` や（道I で得る）`ℝ^p` の上に符号を入れて (p−1,1) 等を
   目指す。

### 次への申し送り
1. **道II を単独で詰める**：符号原理で `T^p`（または `ℝ^p`）にローレンツ符号を入れ、時間方向を
   1つに絞る追加条件を探す（notebook 05 の D は時間方向が複数だった）。
2. **道I を §2-2 の問いに回収**：`cos2θ` から減衰相関が `st`／作用の帰結として出るかを、
   no-go を踏まえて改めて問う。出れば「追加原理」は不要になり、出なければ追加原理の存在が
   確定する。この二択を明確にすることが地図の次の一歩。
